In [ ]:
!pip install pandas

In [ ]:
!pip install dask\[complete\]

In [ ]:
!pip install pyspark

In [ ]:
# 1. Clone your entire DE_Essentials repo into a temporary folder
!git clone https://github.com/franciscocoelh0/DE_Essentials.git temp_repo

# 2. Move the data folder into your new project
!mv temp_repo/data ./data

# 3. Clean up the temporary repository
!rm -rf temp_repo

In [ ]:
!ls -ltr data/nyse_all/nyse_data/

In [ ]:
# Using Pandas 
# Used for lightweight processes. As files are not very big and counter was fairly quick.
# By default uses only one thread (process) to process the data

import glob
import pandas as pd

# Get a list of all CSV files in the directory
files = glob.glob('data/nyse_all/nyse_data/*')
rec_count = 0 

# For each file, read it into a DataFrame and count the number of records
for file in files:
    df = pd.read_csv(
        file,
        names=['stock_id','trans_date','open_price','low_price','high_price', 'close_price','volume']
        )
    rec_count += df.shape[0]

rec_count

9384739

In [ ]:
# Using Dask
# When files are bigger in size. An extension of Pandas as in they run the data processing apps using Pandas style, without running into resource related constraints.  
# Divides the files into multiple chunks and process them in parallel using the full capacity of the machine (distributed-fashion)
# No need to specify chunck size or multiprocessing, Dask will automatically determine it based on the file size and available memory.
# Not very popular when it comes to distributed computing

import dask.dataframe as dd
df = dd.read_csv(
    'data/nyse_all/nyse_data/*',
    names=['stock_id','trans_date','open_price','low_price','high_price','close_price','volume']
    )

df.shape[0].compute()

/Users/franciscocoelho/VSCodeProjects/franciscocoelh0/Random_Scripts/DE_Essentials/apps/data-processing-frameworks/dpf-venv/lib/python3.13/site-packages/dask/dataframe/io/csv.py:505: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(


9384739

In [ ]:
# Using PySpark (compatible with the Spark Engine). Doesn't make sense for lightweight processes.
# It is a Python based library. Works Wll on spark clusters using multiple nodes seamsleys.
# Spark based APIs are typically more developer friend compared to Dask

from pyspark.sql import SparkSession

# When building apps on Databricks this piece of code won't be necessary (it is automatic)
spark = SparkSession. \
    builder. \
    appName('NYSE Count'). \
    master('local'). \
    getOrCreate()


df = spark. \
    read. \
    csv(
        'data/nyse_all/nyse_data/',
        schema = '''
            stock_id STRING, trans_date STRING, open_price STRING,low_price FLOAT,
            high_price FLOAT, close_price FLOAT, volume BIGINT

'''
    )

df.count()

9384739